In [0]:
# récupération du nom du catalog
catalog_name = "labuser_" + spark.sql("SELECT current_user()").collect()[0][0].split("@")[0]

# Définition du schéma à utiliser
schema = "aibi_dashboards"
volume = "source_files"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema}")

# On renseigne les catalogue / schéma à utiliser
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE {schema}")

spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema}.{volume}")

print(f"Catalog : {catalog_name}")
print(f"Schema : {schema}")
print(f"Volume : {volume}")

In [0]:
import os
import pandas as pd

path = os.getcwd() + "/Resources/NB00/"

files = os.listdir(path)
print("Fichiers dans le dossier :", files)

for file in files:
    if file.endswith(".csv"):
        print(f"traitement du fichier {file}")
        
        file_path = f"/Volumes/{catalog_name}/{schema}/{volume}/{file}"
        print(f"Ecriture du fichier dans le volume : {file_path}")

        dbutils.fs.cp(os.path.join(path, file), file_path)
        print("Ecriture du fichier OK")

        df = spark.read.csv(
            f"/Volumes/{catalog_name}/{schema}/{volume}/{file}",
            header=True,
            inferSchema=True
        )

        df.write.mode("overwrite").saveAsTable(file.replace(".csv", ""))
        print("Ecriture dans la table OK")

